<a href="https://colab.research.google.com/github/PrantoHalder/Annonimous_authentification/blob/main/AIoT_Enabled_Smart_Agriculture_Edge_Inference_for_Explainable_Yield_Estimation_and_Irrigation_Recommendation_Using_the_BIED_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix
)

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from xgboost import XGBRegressor, XGBClassifier

# ---------- Load ----------
df = pd.read_csv("/content/BIED_Smart_Agriculture_Dataset.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month

# Realistic sensor-to-decision inputs (exclude downstream labels)
feature_cols = [
    "device_id","location","crop_type","season","hour","day","month",
    "temperature","humidity","rainfall","soil_moisture","soil_ph","light_intensity","fertilizer_used"
]
X = df[feature_cols]
y_yield = df["yield_estimate"]
y_irrig = df["irrigation_needed"].astype(int)

cat_cols = ["device_id","location","crop_type","season"]
num_cols = [c for c in feature_cols if c not in cat_cols]

preprocess = ColumnTransformer([
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median"))
    ]), num_cols)
])

def reg_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": math.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
    }

def clf_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "Acc": accuracy_score(y_true, y_pred),
        "Prec": precision_score(y_true, y_pred),
        "Rec": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
    }, y_pred

# ---------- Models ----------
reg_models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.9, colsample_bytree=0.9,
        reg_lambda=1.0, random_state=42, n_jobs=-1, tree_method="hist"
    ),
}

clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=500, n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.9, colsample_bytree=0.9,
        reg_lambda=1.0, random_state=42, n_jobs=-1, tree_method="hist",
        eval_metric="logloss"
    ),
}

# ---------- Random split ----------
X_train, X_test, y_train_y, y_test_y, y_train_i, y_test_i = train_test_split(
    X, y_yield, y_irrig, test_size=0.2, random_state=42, stratify=y_irrig
)

reg_rows = []
best_reg_name, best_reg_rmse, best_reg_pipe = None, 1e9, None
for name, m in reg_models.items():
    pipe = Pipeline([("prep", preprocess), ("model", m)])
    pipe.fit(X_train, y_train_y)
    pred = pipe.predict(X_test)
    met = reg_metrics(y_test_y, pred)
    reg_rows.append({"Model": name, **met})
    if met["RMSE"] < best_reg_rmse:
        best_reg_rmse, best_reg_name, best_reg_pipe = met["RMSE"], name, pipe

clf_rows = []
best_clf_name, best_clf_f1, best_clf_pipe = None, -1, None
for name, m in clf_models.items():
    pipe = Pipeline([("prep", preprocess), ("model", m)])
    pipe.fit(X_train, y_train_i)
    prob = pipe.predict_proba(X_test)[:, 1]
    met, _ = clf_metrics(y_test_i, prob)
    clf_rows.append({"Model": name, **met})
    if met["F1"] > best_clf_f1:
        best_clf_f1, best_clf_name, best_clf_pipe = met["F1"], name, pipe

reg_df = pd.DataFrame(reg_rows).sort_values("RMSE")
clf_df = pd.DataFrame(clf_rows).sort_values("F1", ascending=False)

reg_df.to_csv("TableIII_yield_random_split.csv", index=False)
clf_df.to_csv("TableIV_irrigation_random_split.csv", index=False)

print("\n=== Table III (Yield, random split) ===")
print(reg_df.to_string(index=False))
print("\n=== Table IV (Irrigation, random split) ===")
print(clf_df.to_string(index=False))
print(f"\nBest yield model: {best_reg_name} (RMSE={best_reg_rmse:.4f})")
print(f"Best irrigation model: {best_clf_name} (F1={best_clf_f1:.4f})")

# ---------- Time split ----------
df_sorted = df.sort_values("timestamp").reset_index(drop=True)
Xt = df_sorted[feature_cols]
yt_y = df_sorted["yield_estimate"]
yt_i = df_sorted["irrigation_needed"].astype(int)
cut = int(0.8 * len(df_sorted))

Xtr, Xte = Xt.iloc[:cut], Xt.iloc[cut:]
ytr_y, yte_y = yt_y.iloc[:cut], yt_y.iloc[cut:]
ytr_i, yte_i = yt_i.iloc[:cut], yt_i.iloc[cut:]

reg_time_pipe = Pipeline([("prep", preprocess), ("model", reg_models[best_reg_name])])
reg_time_pipe.fit(Xtr, ytr_y)
pred_time = reg_time_pipe.predict(Xte)
time_reg = reg_metrics(yte_y, pred_time)

clf_time_pipe = Pipeline([("prep", preprocess), ("model", clf_models[best_clf_name])])
clf_time_pipe.fit(Xtr, ytr_i)
prob_time = clf_time_pipe.predict_proba(Xte)[:, 1]
time_clf, _ = clf_metrics(yte_i, prob_time)

pd.DataFrame([{"BestYieldModel": best_reg_name, **time_reg}]).to_csv("TimeSplit_yield.csv", index=False)
pd.DataFrame([{"BestIrrigationModel": best_clf_name, **time_clf}]).to_csv("TimeSplit_irrigation.csv", index=False)

print("\n=== Time split summary ===")
print("Yield:", time_reg)
print("Irrigation:", time_clf)

# ---------- Figures ----------
# Fig 2: yield histogram
plt.figure(figsize=(6.6, 3.2))
plt.hist(df["yield_estimate"], bins=30)
plt.xlabel("Yield estimate")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("Fig2_yield_hist.png", dpi=300)
plt.close()

# Fig 3: irrigation-needed rate by crop
irrig_by_crop = df.groupby("crop_type")["irrigation_needed"].mean().sort_values(ascending=False)
plt.figure(figsize=(6.6, 3.2))
plt.bar(irrig_by_crop.index, irrig_by_crop.values)
plt.xlabel("Crop type")
plt.ylabel("Irrigation-needed rate")
plt.tight_layout()
plt.savefig("Fig3_irrigation_by_crop.png", dpi=300)
plt.close()

# Fig 4: confusion matrix for best irrigation model (random split)
best_prob = best_clf_pipe.predict_proba(X_test)[:, 1]
_, best_pred = clf_metrics(y_test_i, best_prob)
cm = confusion_matrix(y_test_i, best_pred)

plt.figure(figsize=(4.2, 3.2))
plt.imshow(cm)
plt.xticks([0, 1], ["No", "Yes"])
plt.yticks([0, 1], ["No", "Yes"])
plt.xlabel("Predicted")
plt.ylabel("True")
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.tight_layout()
plt.savefig("Fig4_confusion_matrix.png", dpi=300)
plt.close()

# Fig 5: SHAP summary for best yield model (optional; needs shap)
try:
    import shap
    X_sample = X_test.sample(n=min(600, len(X_test)), random_state=42)

    Xt_enc = best_reg_pipe.named_steps["prep"].transform(X_sample)
    model = best_reg_pipe.named_steps["model"]

    oh = best_reg_pipe.named_steps["prep"].named_transformers_["cat"].named_steps["oh"]
    cat_names = oh.get_feature_names_out(cat_cols)
    feat_names = list(cat_names) + num_cols

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(Xt_enc)

    plt.figure()
    shap.summary_plot(shap_values, Xt_enc, feature_names=feat_names, show=False)
    plt.tight_layout()
    plt.savefig("Fig5_shap_summary.png", dpi=300, bbox_inches="tight")
    plt.close()
except Exception as e:
    print("SHAP skipped or failed:", type(e).__name__, str(e))


=== Table III (Yield, random split) ===
        Model      MAE     RMSE       R2
Random Forest 0.570346 0.951406 0.994489
      XGBoost 0.590861 0.973030 0.994235
        Ridge 1.546883 2.399173 0.964953

=== Table IV (Irrigation, random split) ===
              Model    Acc     Prec      Rec       F1  ROC-AUC
            XGBoost 0.9995 0.997253 1.000000 0.998624 0.999998
      Random Forest 0.9995 1.000000 0.997245 0.998621 1.000000
Logistic Regression 0.9685 0.928571 0.895317 0.911641 0.991498

Best yield model: Random Forest (RMSE=0.9514)
Best irrigation model: XGBoost (F1=0.9986)

=== Time split summary ===
Yield: {'MAE': 0.5915480166666662, 'RMSE': 1.013217325169624, 'R2': 0.9935106177591537}
Irrigation: {'Acc': 0.9975, 'Prec': 0.996268656716418, 'Rec': 0.994413407821229, 'F1': 0.9953401677539608, 'ROC-AUC': np.float64(0.9999834527914504)}
SHAP skipped or failed: UFuncTypeError Cannot cast ufunc 'isnan' input from dtype('O') to dtype('bool') with casting rule 'same_kind'
